# 02 Feature Engineering

Start from the clean dataset, add a handful of dependable indicators, and save a feature-ready table for modeling strategies, and backtests

In [1]:
import os
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.settings import load_config
from src.utils.logger import get_logger
from src.data.data_manager import DataManager
from src.indicators.moving_average import calculate_sma
from src.indicators.momentum_indicators import calculate_rsi

config = load_config()
logger = get_logger('notebooks.feature_engineering')
data_manager = DataManager(config=config)


Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
Loading cofiguration from /home/ciprimarian/Repos/ Local Repos/tradingbot/src/config/trading_config.yaml...
2025-11-18 19:07:32,742 | INFO | src.data.market_data | Market Data handler initialized | base_url: https://data.alpaca.markets


## Load cleaned baseline

In [2]:
df_clean = data_manager.load_data('cleaned_data')
if df_clean.empty:
    raise RuntimeError('Run 01_data_exploration_and_cleaning.ipynb to generate cleaned data before proceeding.')

df_features = df_clean.copy()
df_features.tail()

2025-11-18 19:07:35,220 | INFO | src.data.data_manager | Loading dataset cleaned_data from data/processed/cleaned_data.parquet


,open,high,low,close,volume
t,,,,,
2024-05-17 04:00:00+00:00,943.69,947.4,918.06,924.79,35989353
2024-05-20 04:00:00+00:00,937.50,952.0,934.40,947.80,31876446
2024-05-21 04:00:00+00:00,935.99,954.0,931.80,953.86,32894646
2024-05-22 04:00:00+00:00,954.59,960.2,932.49,949.50,54865849
2024-05-23 04:00:00+00:00,1020.28,1063.2,1015.20,1037.99,83506528


## Build a minimal feature set

In [4]:
fast = config['backtest']['strategy_params'].get('fast_period', 20)
slow = config['backtest']['strategy_params'].get('slow_period', 50)

calculate_sma(df_features, period=fast)
calculate_sma(df_features, period=slow)
df_features['rsi_14'] = calculate_rsi(df_features, window=14)
df_features['return_1d'] = df_features['close'].pct_change()
df_features['lag_close_1'] = df_features['close'].shift(1)
df_features['day_of_week'] = df_features.index.dayofweek

df_features = df_features.dropna()
logger.info("Featured rows ater dropna: %s", len(df_features))

path = data_manager.save_data(df_features, "engineered_features")
print(f"Feature-engineered data saved to: {path}")
df_features.head()

2025-11-18 19:18:22,146 | INFO | notebooks.feature_engineering | Featured rows ater dropna: 98
2025-11-18 19:18:22,148 | INFO | src.data.data_manager | Saving dataset engineered_features to data/processed/engineered_features.parquet
Feature-engineered data saved to: data/processed/engineered_features.parquet


,open,high,low,close,volume,sma_20,sma_50,rsi_14,return_1d,lag_close_1,day_of_week
t,,,,,,,,,,,
2024-01-04 05:00:00+00:00,477.67,485.00,475.0800,479.98,30662544,477.835000,477.835000,96.655544,0.009018,475.69,3
2024-01-05 05:00:00+00:00,484.62,495.47,483.0601,490.97,41503927,482.213333,482.213333,96.655544,0.022897,479.98,4
2024-01-08 05:00:00+00:00,495.12,522.75,494.7900,522.53,64308094,492.292500,492.292500,96.655544,0.064281,490.97,0
2024-01-09 05:00:00+00:00,524.01,543.25,516.9000,531.40,77334975,500.114000,500.114000,96.655544,0.016975,522.53,1
2024-01-10 05:00:00+00:00,536.16,546.00,534.8900,543.50,53394953,507.345000,507.345000,96.655544,0.022770,531.40,2
